In [14]:
# Imports and Data
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import stopwords

In [15]:
df = pd.read_csv('IMDB Dataset.csv')

In [16]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [17]:
df.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [18]:
df.describe()

,review,sentiment
count,50000,50000
unique,49582,2
top,Loved today's show!!! It was a variety and not...,positive
freq,5,25000


In [20]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [22]:
df.duplicated().sum()

np.int64(418)

In [23]:
# Sample Data (In a real project, load this from a file)
# texts = [
#     "I love this movie, it's fantastic!",
#     "This was a terrible film.",
#     "The acting was superb and the plot was great.",
#     "I would not recommend this to anyone.",
#     "It was an okay movie, not the best but enjoyable.",
#     "Absolutely brilliant, a must-see!",
#     "A complete waste of time and money.",
#     "The story was compelling and engaging."
# ]

In [24]:
# Labels- 1 for Positive, 0 for Negative
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

In [25]:
# Drop duplicates
df = df.drop_duplicates(subset=['review']).reset_index(drop=True)

In [26]:
df.duplicated().sum()

np.int64(0)

In [28]:
# Text Preprocessing Function
stop_words = set(stopwords.words('english')) - {'not','no','nor','neither','never'}

def preprocess_text(text):
    # Make text lowercase
    text = text.lower()
    #remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Tokenize and remove stopwords
    tokens = text.split()
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return " ".join(filtered_tokens)

# Apply preprocessing to our dataset
df['processed_review'] = df['review'].apply(preprocess_text) 
print("--- Original vs. Processed ---")
for i in range(3):
    print(f"Original: {texts[i]}")
    print(f"Processed: {processed_texts[i]}\n")

--- Original vs. Processed ---
Original: I love this movie, it's fantastic!
Processed: love movie fantastic

Original: This was a terrible film.
Processed: terrible film

Original: The acting was superb and the plot was great.
Processed: acting superb plot great



In [29]:
# Splitting data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    df['processed_review'], 
    df['sentiment'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df['sentiment']
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 39665
Testing samples: 9917


In [30]:
#feature extraction (vectorization)
# Initialize the TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

# Fit the vectorizer on the training data and transform it
X_train_tfidf = vectorizer.fit_transform(X_train)

# Only transform the test data using the already-fitted vectorizer
X_test_tfidf = vectorizer.transform(X_test)

print("Shape of training data vectors:", X_train_tfidf.shape)
print("Shape of testing data vectors:", X_test_tfidf.shape)

Shape of training data vectors: (39665, 10000)
Shape of testing data vectors: (9917, 10000)


In [31]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression often yields ~88-90% accuracy on IMDB
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

Accuracy: 0.8924069779167086
              precision    recall  f1-score   support

    Negative       0.90      0.88      0.89      4940
    Positive       0.89      0.90      0.89      4977

    accuracy                           0.89      9917
   macro avg       0.89      0.89      0.89      9917
weighted avg       0.89      0.89      0.89      9917



In [32]:
#training the NLP model
# Initialize and train the Naive Bayes classifier
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

print("Model training complete.")

Model training complete.


In [33]:
# Make predictions on the test set
y_pred = model.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")

# Display a detailed classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))


Model Accuracy: 86.82%

Classification Report:
              precision    recall  f1-score   support

    Negative       0.88      0.85      0.87      4940
    Positive       0.86      0.88      0.87      4977

    accuracy                           0.87      9917
   macro avg       0.87      0.87      0.87      9917
weighted avg       0.87      0.87      0.87      9917



In [38]:
#testing the model on new senences
# Function to predict sentiment of a new sentence
def log_reg_model(sentence):
    # Preprocess the text
    processed_sentence = preprocess_text(sentence)
    
    # Vectorize the text using the SAME vectorizer
    vectorized_sentence = vectorizer.transform([processed_sentence])
    
    # Make a prediction
    prediction = model.predict(vectorized_sentence)
    
    # Return the result
    return "Positive" if prediction[0] == 1 else "Negative"

# Test with new sentences
new_sentence_1 = "The movie was absolutely amazing!"
new_sentence_2 = "I was very bored and did not like it."

print(f"'{new_sentence_1}' -> Sentiment: {predict_sentiment(new_sentence_1)}")
print(f"'{new_sentence_2}' -> Sentiment: {predict_sentiment(new_sentence_2)}")


'The movie was absolutely amazing!' -> Sentiment: Positive
'I was very bored and did not like it.' -> Sentiment: Negative


In [39]:
# Logistic Regression Model train 
log_reg_model = LogisticRegression(max_iter=1000)
log_reg_model.fit(X_train_tfidf, y_train)

#  Logistic Regression use prediction function
def predict_sentiment_logreg(sentence):
    processed = preprocess_text(sentence)
    vectorized = vectorizer.transform([processed])
    prediction = log_reg_model.predict(vectorized)[0]
    return "Positive" if prediction == 1 else "Negative"

# Check / Test karein
print(predict_sentiment_logreg("The movie was fantastic and brilliant!"))
print(predict_sentiment_logreg("Worst movie ever, completely wasted my time."))

Positive
Negative


In [40]:
import joblib

# Model aur Vectorizer save karein
joblib.dump(log_reg_model, 'sentiment_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!
